In [1]:
import os
os.chdir("..")
print(os.getcwd())

/Users/jacoboromerodiaz/Projects/video-retrieval-cvpr


In [4]:
import torch
import torch.nn.functional as F
from pathlib import Path
from covr.data.dataset import RetrievalDataset
from covr.models.cross_attention import CrossAttentionFusion, GalleryEncoder, FusionConfig

CKPT = Path("checkpoints/cross_attn_dev/ckpt_epoch0005.pt")
GAL_DIR = Path("features/gallery/vjepa_base_val")
QUERY_DIR = Path("features/queries/flan_large")
JSON_PATH = Path("covr/data/covr-r/merged_webvid_ss2.json")
DEVICE = torch.device("cpu")
MAX_PATCHES = 512

fusion_cfg = FusionConfig(text_dim=1024)
fusion = CrossAttentionFusion(fusion_cfg).to(DEVICE)
gallery_enc = GalleryEncoder(fusion_cfg).to(DEVICE)

ckpt = torch.load(CKPT, map_location=DEVICE, weights_only=True)
fusion.load_state_dict(ckpt["fusion"])
gallery_enc.load_state_dict(ckpt["gallery_enc"])
fusion.eval(); gallery_enc.eval()
print(f"loaded checkpoint  epoch={ckpt['epoch']}")

dataset = RetrievalDataset(JSON_PATH, video_root="", load_frames=False)
available_gal = {p.stem for p in GAL_DIR.glob("*.pt")}
available_query = {p.stem for p in QUERY_DIR.glob("*.pt")}

sample = None
for item in dataset:
    if (item["source_video_id"] in available_gal
            and item["target_video_id"] in available_gal
            and item["source_video_id"] in available_query):
        sample = item
        break
assert sample is not None, "no valid sample found"
print(f"\nsource: {sample['source_video_id']}")
print(f"target: {sample['target_video_id']}")
print(f"query: {sample['modification_text'][:120]}...")

def _load(path: Path) -> torch.Tensor:
    t = torch.load(path, weights_only=True)
    return t[:MAX_PATCHES] if t.shape[0] > MAX_PATCHES else t

src_patches = _load(GAL_DIR   / f"{sample['source_video_id']}.pt").unsqueeze(0).to(DEVICE)
query_emb = _load(QUERY_DIR / f"{sample['source_video_id']}.pt").unsqueeze(0).to(DEVICE)

with torch.no_grad():
    q_vec = fusion(src_patches, query_emb)          # [1, embed_dim]

gal_paths = sorted(GAL_DIR.glob("*.pt"))
print(f"\nbuilding gallery index over {len(gal_paths)} videos …", end=" ", flush=True)
gallery_ids, gallery_vecs = [], []
with torch.no_grad():
    for p in gal_paths:
        t = _load(p)
        if torch.isnan(t).any():
            continue
        gallery_ids.append(p.stem)
        gallery_vecs.append(gallery_enc(t.unsqueeze(0).to(DEVICE)).squeeze(0))
print("done")

G = torch.stack(gallery_vecs) # [N, embed_dim]
sims = (q_vec @ G.T).squeeze(0) # [N]
ranked = sims.argsort(descending=True).tolist()

target_id = sample["target_video_id"]
target_rank = next((r + 1 for r, i in enumerate(ranked) if gallery_ids[i] == target_id), None)
print(f"\nground-truth target rank: {target_rank} / {len(gallery_ids)}")
print(f"R@1={int(target_rank == 1)}  R@5={int(target_rank <= 5)}",
      f"R@10={int(target_rank <= 10)} R@50={int(target_rank <= 50)}")

print("\nTop-5 results:")
for rank, idx in enumerate(ranked[:5], 1):
    marker = " ← GT" if gallery_ids[idx] == target_id else ""
    print(f"{rank}. id={gallery_ids[idx]}  sim={sims[idx]:.4f}{marker}")


loaded checkpoint  epoch=5


AssertionError: no valid sample found

In [9]:
import yaml
import torch
from pathlib import Path
from torch.utils.data import random_split

from covr.evaluation.metrics import recall_at_k
from covr.models.cross_attention import CrossAttentionFusion, GalleryEncoder, FusionConfig
from scripts.train_cross_attention import TrainDataset

# ── config (same source of truth as training) ─────────────────────────────────
with open("configs/fusion/cross_attention_dev.yaml") as f:
    cfg = yaml.safe_load(f)

CKPT        = Path("checkpoints/cross_attn_dev/ckpt_epoch0005.pt")
GAL_DIR     = Path(cfg["embeddings_dir"])   # features/gallery/vjepa_base
QRY_DIR     = Path(cfg["queries_dir"])      # features/queries/flan_large
DEVICE      = torch.device("cpu")
MAX_PATCHES = cfg.get("max_patches", 512)

# ── model ─────────────────────────────────────────────────────────────────────
fusion_cfg  = FusionConfig(text_dim=cfg["text_dim"])
fusion      = CrossAttentionFusion(fusion_cfg).to(DEVICE)
gallery_enc = GalleryEncoder(fusion_cfg).to(DEVICE)

ckpt = torch.load(CKPT, map_location=DEVICE, weights_only=True)
fusion.load_state_dict(ckpt["fusion"])
gallery_enc.load_state_dict(ckpt["gallery_enc"])
fusion.eval(); gallery_enc.eval()
print(f"loaded checkpoint  epoch={ckpt['epoch']}")

# ── replicate exact train/val split from training script ──────────────────────
full_ds = TrainDataset(
    cfg["json_path"],
    cfg["embeddings_dir"],
    cfg["queries_dir"],
    split=cfg.get("split", "all"),
    dev=cfg["mode"] == "dev",
    max_patches=MAX_PATCHES,
)
val_frac = cfg.get("val_frac", 0.1)
val_size = max(1, int(len(full_ds) * val_frac))
_, val_ds = random_split(
    full_ds,
    [len(full_ds) - val_size, val_size],
    generator=torch.Generator().manual_seed(42),
)
# Recover raw metadata (video IDs) for val samples without loading tensors
val_meta = [
    full_ds.inner.samples[full_ds.indices[j]]
    for j in val_ds.indices
]
print(f"val set: {len(val_meta)} samples (val_frac={val_frac}, seed=42)")

# ── build full gallery index ───────────────────────────────────────────────────
def _load(path: Path) -> torch.Tensor:
    t = torch.load(path, weights_only=True)
    return t[:MAX_PATCHES] if t.shape[0] > MAX_PATCHES else t

gal_paths = sorted(GAL_DIR.glob("*.pt"))
print(f"building gallery over {len(gal_paths)} videos …", end=" ", flush=True)
gallery_ids, gallery_vecs = [], []
with torch.no_grad():
    for p in gal_paths:
        t = _load(p)
        if torch.isnan(t).any():
            continue
        gallery_ids.append(p.stem)
        gallery_vecs.append(gallery_enc(t.unsqueeze(0).to(DEVICE)).squeeze(0))
print("done")
G = torch.stack(gallery_vecs)                       # [N_gal, embed_dim]
gal_id_to_idx = {vid: i for i, vid in enumerate(gallery_ids)}

# ── compute query vectors for val samples ─────────────────────────────────────
q_vecs, gt_indices = [], []
skipped = 0
for meta in val_meta:
    src_id = str(meta["video_source"]).split("/")[-1]
    tgt_id = str(meta["video_target"]).split("/")[-1]
    src_path = GAL_DIR / f"{src_id}.pt"
    qry_path = QRY_DIR / f"{src_id}.pt"
    if not src_path.exists() or not qry_path.exists() or tgt_id not in gal_id_to_idx:
        skipped += 1
        continue
    with torch.no_grad():
        q = fusion(_load(src_path).unsqueeze(0).to(DEVICE),
                   _load(qry_path).unsqueeze(0).to(DEVICE))
    q_vecs.append(q.squeeze(0).cpu())
    gt_indices.append(gal_id_to_idx[tgt_id])

if skipped:
    print(f"skipped {skipped} samples (missing .pt files)")

Q      = torch.stack(q_vecs)       # [N_eval, embed_dim]
scores = Q @ G.T                   # [N_eval, N_gal]
gt     = torch.tensor(gt_indices)

print(f"\n=== Val metrics ({len(Q)} samples, gallery={len(gallery_ids)}) ===")
print(f"R@1   = {recall_at_k(scores, gt,  1):.4f}")
print(f"R@5   = {recall_at_k(scores, gt,  5):.4f}")
print(f"R@10  = {recall_at_k(scores, gt, 10):.4f}")
print(f"R@50  = {recall_at_k(scores, gt, 50):.4f}")


loaded checkpoint  epoch=5
all
[dev] 1459/2634 pairs available
val set: 145 samples (val_frac=0.1, seed=42)
building gallery over 2568 videos … done

=== Val metrics (145 samples, gallery=2568) ===
R@1   = 0.0345
R@5   = 0.2138
R@10  = 0.3172
R@50  = 0.6000
